# 表の数値から文章を作り、数値の誤りを検査する（data-to-text）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_summarization/data_to_text.ipynb)

NLP クックブック 28-17 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

試合結果や決算のような表のデータから、短い記事の文章を作りたい。生成した文章の数値や名前が表と合っているかを自動で確かめ、テンプレートで作る方法とも比べたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import numpy as np

# 架空の試合結果と決算（このレシピのために乱数で作ったもの。チーム・会社・人名は実在しない）
rng = np.random.default_rng(0)
stadiums = {"北浜ペリカンズ": "北浜スタジアム", "南町ラクーンズ": "南町球場", "東川ハミングバーズ": "東川ドーム",
            "西原バイソンズ": "西原パーク", "青葉コメッツ": "青葉フィールド", "白砂タートルズ": "白砂球場"}
surnames = ["佐藤", "高橋", "田中", "伊藤", "渡辺", "山本", "中村", "小林", "加藤", "吉田", "山田", "松本", "井上", "木村", "清水", "森", "池田", "橋本"]
roster = {t: list(rng.choice(surnames, 3, replace=False)) for t in stadiums}

games = []
for _ in range(12):
    home, away = (str(t) for t in rng.choice(list(stadiums), 2, replace=False))
    hs, vs = (int(x) for x in rng.choice(10, 2, replace=False))      # 引き分けはなし
    win, lose = (home, away) if hs > vs else (away, home)
    hr = str(rng.choice([home, away]))
    games.append({"日付": f"{rng.integers(4, 10)}月{rng.integers(1, 29)}日", "球場": stadiums[home], "ホーム": home, "ビジター": away,
                  "ホームの得点": hs, "ビジターの得点": vs, "勝利投手": f"{roster[win][0]}（{win}）", "敗戦投手": f"{roster[lose][0]}（{lose}）",
                  "本塁打": f"{roster[hr][1]}（{hr}）{rng.integers(1, 25)}号", "観客数": int(rng.integers(8000, 45000))})

companies = ["北浜精機", "みなと食品", "青葉電子", "さくら物流", "東雲化学", "ひかり通信", "若葉製薬", "白砂建設", "南町鉄工", "つばさ商事", "西原紙業", "東川電機"]
reasons_up, reasons_down = ["海外向けの販売が好調だった", "新製品の販売が伸びた", "値上げが浸透した"], ["原材料費が高騰した", "国内の需要が低迷した", "円高で海外の売上が目減りした"]
earnings = []
for i, name in enumerate(companies):
    rev0 = int(rng.integers(200, 5000)); rev1 = int(round(rev0 * (1 + rng.normal(0.03, 0.10))))
    op0 = int(round(rev0 * rng.uniform(0.03, 0.12))); op1 = int(round(op0 * (1 + rng.normal(0, 0.3))))
    if i == 3: op1 = -int(round(op0 * 0.4))                        # 赤字に転じた会社と、黒字に戻った会社を 1 社ずつ入れる
    if i == 7: op0, op1 = -int(round(op0 * 0.5)), int(round(op0 * 0.6))
    earnings.append({"会社名": name, "決算期": "2025年度上期", "売上高（億円）": rev1, "前年同期の売上高（億円）": rev0,
                     "営業利益（億円）": op1, "前年同期の営業利益（億円）": op0,
                     "主な要因": str(rng.choice(reasons_up if op1 > op0 else reasons_down))})

def to_table(rec):
    return "\n".join(f"{k}: {v}" for k, v in rec.items())

print(to_table(games[0]), "\n")
print(to_table(earnings[3]))

日付: 5月14日
球場: 南町球場
ホーム: 南町ラクーンズ
ビジター: 青葉コメッツ
ホームの得点: 7
ビジターの得点: 0
勝利投手: 橋本（南町ラクーンズ）
敗戦投手: 木村（青葉コメッツ）
本塁打: 高橋（南町ラクーンズ）11号
観客数: 22919 

会社名: さくら物流
決算期: 2025年度上期
売上高（億円）: 762
前年同期の売上高（億円）: 772
営業利益（億円）: -36
前年同期の営業利益（億円）: 90
主な要因: 円高で海外の売上が目減りした


In [3]:
# 1. テンプレートによる生成: 表の値を文の型に流し込む。計算（点差や前年同期比）はコードで行う
def pct(cur, prev):
    return (cur - prev) / abs(prev) * 100

def template_game(g):
    hs, vs = g["ホームの得点"], g["ビジターの得点"]
    win, lose = (g["ホーム"], g["ビジター"]) if hs > vs else (g["ビジター"], g["ホーム"])
    return (f"{g['日付']}、{g['球場']}で行われた{g['ホーム']}対{g['ビジター']}の試合は、{win}が{max(hs, vs)}対{min(hs, vs)}で{lose}に勝った。"
            f"勝利投手は{g['勝利投手'].split('（')[0]}、敗戦投手は{g['敗戦投手'].split('（')[0]}。本塁打は{g['本塁打']}。観客数は{g['観客数']:,}人だった。")

def template_earnings(e):
    r1, r0, o1, o0 = e["売上高（億円）"], e["前年同期の売上高（億円）"], e["営業利益（億円）"], e["前年同期の営業利益（億円）"]
    p = pct(r1, r0)
    s = f"{e['会社名']}の{e['決算期']}の売上高は{r1:,}億円で、前年同期比{abs(p):.1f}%の{'増収' if r1 > r0 else '減収'}だった。"
    if o1 < 0 <= o0:
        s += f"営業損益は{-o1:,}億円の赤字（前年同期は{o0:,}億円の黒字）に転落した。"
    elif o0 < 0 <= o1:
        s += f"営業損益は{o1:,}億円の黒字（前年同期は{-o0:,}億円の赤字）に転換した。"
    else:
        s += f"営業利益は{o1:,}億円で、前年同期比{abs(pct(o1, o0)):.1f}%の{'増益' if o1 > o0 else '減益'}となった。"
    return s + f"{e['主な要因']}ことが影響した。"

records = [("試合", g) for g in games] + [("決算", e) for e in earnings]
template_texts = [template_game(r) if kind == "試合" else template_earnings(r) for kind, r in records]
print(template_texts[0], "\n")
print(template_texts[12 + 3])

5月14日、南町球場で行われた南町ラクーンズ対青葉コメッツの試合は、南町ラクーンズが7対0で青葉コメッツに勝った。勝利投手は橋本、敗戦投手は木村。本塁打は高橋（南町ラクーンズ）11号。観客数は22,919人だった。 

さくら物流の2025年度上期の売上高は762億円で、前年同期比1.3%の減収だった。営業損益は36億円の赤字（前年同期は90億円の黒字）に転落した。円高で海外の売上が目減りしたことが影響した。


In [4]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # この章のほかのレシピと同じ 3B の日本語モデル（MIT ライセンス）。VRAM 約 7GB
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(prompt, max_new_tokens=200):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def with_computed(kind, r):
    """方法 B: 計算が要る値（勝敗、点差、前年同期比、増収・減収）をコードで求めて表に足す"""
    r = dict(r)
    if kind == "試合":
        hs, vs = r["ホームの得点"], r["ビジターの得点"]
        r["勝ったチーム"], r["点差"] = (r["ホーム"] if hs > vs else r["ビジター"]), abs(hs - vs)
    else:
        r1, r0, o1, o0 = r["売上高（億円）"], r["前年同期の売上高（億円）"], r["営業利益（億円）"], r["前年同期の営業利益（億円）"]
        r["売上高の前年同期比"] = f"{abs(pct(r1, r0)):.1f}%{'増' if r1 > r0 else '減'}（{'増収' if r1 > r0 else '減収'}）"
        if (o1 < 0) != (o0 < 0):
            r["営業損益の変化"] = "赤字に転落" if o1 < 0 else "黒字に転換"
        else:
            r["営業利益の前年同期比"] = f"{abs(pct(o1, o0)):.1f}%{'増' if o1 > o0 else '減'}（{'増益' if o1 > o0 else '減益'}）"
    return r

def prompt(kind, r):
    what = "プロ野球の試合結果" if kind == "試合" else "企業の決算"
    return (f"次の表は{what}のデータです。この内容を伝える 2〜3 文のニュース記事を書いてください。"
            f"表にない情報は書かないでください。記事だけを出力してください。\n\n{to_table(r)}")

llm_texts = {"LLM（表だけ）": [chat(prompt(k, r)) for k, r in records],                       # T4 で 24 件に数分
             "LLM（計算済みの値も渡す）": [chat(prompt(k, with_computed(k, r))) for k, r in records]}
for method, texts in llm_texts.items():
    print(f"■ {method}\n{texts[0]}\n\n{texts[12 + 3]}\n")

■ LLM（表だけ）
南町球場で5月14日に開催されたプロ野球試合において、南町ラクーンズが青葉コメッツを7対0で下した。ホームチームの南町ラクーンズは、勝利投手の橋本投手の活躍と高橋選手の11号本塁打により、鮮やかな勝利を収めた。この試合、青葉コメッツは完封負けを喫し、敗戦投手は木村投手となった。観客数は2万2919人を集め、球場は大いに盛り上がった。

さくら物流は2025年度上期の決算を発表し、売上高は前年同期比で微減となる762億円であった。特に、海外売上が円高の影響により目減りしたことが主な要因として挙げられている。一方、営業利益は-36億円と赤字に転落し、前年同期の90億円から大きく悪化した。同社はコスト削減や事業戦略の見直しにより、業績改善に努める方針である。

■ LLM（計算済みの値も渡す）
南町球場で行われたプロ野球試合において、南町ラクーンズが青葉コメッツを7対0で下し、勝利を収めた。ホームチームの南町ラクーンズは、橋本投手が勝利投手となり、高橋選手が11号本塁打を放つ活躍を見せた。試合は7対0の大差で進み、観客数は22,919人を記録した。

さくら物流は2025年度上期の決算を発表し、売上高は762億円となり、前年同期から1.3%減少した。特に、円高の影響により海外売上が目減りしたことが主な要因で、前年同期の売上高772億円から減収となった。また、営業利益は-36億円と赤字に転落し、前年同期の90億円から大きく悪化した。これにより、同社は営業損益の改善が急務となっている。



In [5]:
import re
import unicodedata
from fugashi import Tagger

tagger = Tagger()
UNIT = {"兆": 1e12, "億": 1e8, "万": 1e4, "千": 1e3}

def norm(text):
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"(?<=\d)[, ](?=\d)", "", text)                  # 「1,234」「1 234」の区切りを取る

def numbers(text):
    """文中の数値を取り出す。「3万2千人」「1兆234億円」のような単位の組み合わせは 1 つの値にまとめる。符号は見ない"""
    out = []
    for m in re.finditer(r"(?:\d+(?:\.\d+)?[兆億万千]?)+", norm(text)):
        s, value = m.group(), 0.0
        for num, unit in re.findall(r"(\d+(?:\.\d+)?)([兆億万千]?)", s):
            value += float(num) * UNIT.get(unit, 1)
        out.append((s, value, re.search(r"(約|およそ|ほぼ|近く)$", norm(text)[:m.start()]) is not None))
    return out

def allowed_values(kind, r):
    """表から言える値: 表の数値そのもの（億円の値は円にした値も）と、そこから計算できる値（点差、合計、増減額、前年同期比）"""
    exact = {float(x) for v in r.values() for x in re.findall(r"\d+", norm(str(v)))}
    derived = set()
    if kind == "試合":
        hs, vs = r["ホームの得点"], r["ビジターの得点"]
        exact |= {abs(hs - vs), hs + vs}
    else:
        r1, r0, o1, o0 = r["売上高（億円）"], r["前年同期の売上高（億円）"], r["営業利益（億円）"], r["前年同期の営業利益（億円）"]
        oku = [abs(x) for x in (r1, r0, o1, o0, r1 - r0, o1 - o0)]
        exact |= set(oku) | {x * 1e8 for x in oku}
        derived |= {abs(pct(r1, r0)), abs(pct(o1, o0)), r1 / r0 * 100, abs(o1 / o0) * 100 if o0 else 0}
    return exact, derived

def supported(s, x, approx, exact, derived):
    if any(abs(x - v) < 1e-6 for v in exact):
        return True
    decimals = len(s.split(".")[1].rstrip("兆億万千")) if "." in s else 0
    if any(round(d, decimals) == x for d in derived):               # 計算した値を丸めて書いたもの（12.34% → 12.3%）
        return True
    if approx and any(abs(x - v) <= 0.05 * v for v in exact | derived):   # 「約」が付いていれば 5% までのずれを許す
        return True
    zeros = len(str(int(x))) - len(str(int(x)).rstrip("0")) if x == int(x) and x >= 1000 else 0   # 「3万2千人」のような概数は、
    return zeros > 0 and any(round(v / 10 ** zeros) * 10 ** zeros == x and abs(x - v) <= 0.05 * v for v in exact)   # 丸めた値が一致し、ずれが 5% 以内なら認める

def numeric_errors(kind, r, text):
    exact, derived = allowed_values(kind, r)
    return [f"表にない数値 {s}" for s, x, approx in numbers(text) if not supported(s, x, approx, exact, derived)]

def name_errors(r, text):
    """固有名詞（形態素解析で固有名詞とされた語）のうち、表のどこにも書かれていないもの"""
    table = norm(" ".join(str(v) for v in r.values()))
    return [f"表にない固有名詞 {w.surface}" for w in tagger(norm(text))
            if w.feature.pos2 == "固有名詞" and w.surface not in table]

WIN, LOSE = r"勝ち(?!投手|越)|勝利(?!投手)|勝っ|勝つ|制し|下し|破っ|破り|白星", r"敗れ|負け|敗戦(?!投手)|黒星"
UP, DOWN = r"増加|増え|伸び|上回|拡大|改善|黒字に転|黒字転|黒字化", r"減少|減っ|下回|落ち込|縮小|悪化|赤字に転|赤字転|赤字化|転落"

def logic_errors(kind, r, text):
    """勝ち負けや増減の向きが表と合っているか"""
    errs, t = [], norm(text)
    if kind == "試合":
        hs, vs = r["ホームの得点"], r["ビジターの得点"]
        win, lose = (r["ホーム"], r["ビジター"]) if hs > vs else (r["ビジター"], r["ホーム"])
        for a, b in re.findall(r"(\d+)\s*(?:対|-|−|–)\s*(\d+)", t):
            if sorted([int(a), int(b)]) != sorted([hs, vs]):
                errs.append(f"スコアが違う {a}対{b}")
        for sent in re.split(r"(?<=。)", t):
            for verb, should in [(WIN, win), (LOSE, lose)]:
                v = re.search(verb, sent)
                if not v:
                    continue
                subj = [(m.start(), team) for team in (win, lose) for m in re.finditer(f"(?:{team}|{team[2:]})[がは]", sent[:v.start()])]
                if subj and min(subj)[1] != should:
                    errs.append(f"勝敗が逆 「{sent[:40]}」")
    else:
        truth = {"売上": r["売上高（億円）"] > r["前年同期の売上高（億円）"], "利益": r["営業利益（億円）"] > r["前年同期の営業利益（億円）"]}
        for sent in re.split(r"(?<=。)", t):
            claims = [("売上" if w[1] == "収" else "利益", w[0] == "増", w) for w in re.findall(r"[増減][収益]", sent)]
            keys = list(re.finditer(r"売上高?|営業利益|営業損益|営業損失|利益", sent))
            for j, k in enumerate(keys):          # 「売上高は…」から次の項目名か文末までに、増減の語があるか
                seg = sent[k.end(): keys[j + 1].start() if j + 1 < len(keys) else len(sent)]
                up, down = re.search(UP, seg), re.search(DOWN, seg)
                if up or down:
                    claims.append(("売上" if "売上" in k.group() else "利益", down is None or (up is not None and up.start() < down.start()), k.group() + seg[:25]))
            errs += [f"{metric}の増減が逆 「{where}」" for metric, said_up, where in claims if said_up != truth[metric]]
    return sorted(set(errs), key=errs.index)

def check(kind, r, text):
    return numeric_errors(kind, r, text) + name_errors(r, text) + logic_errors(kind, r, text)

# 検査器の確認 1: テンプレートの文（表と合っている）で誤検出がないか
print("テンプレートで誤りとされたもの:", [(i, e) for i, ((k, r), t) in enumerate(zip(records, template_texts)) for e in check(k, r, t)])

# 検査器の確認 2: テンプレートの文にわざと誤りを入れて、見つけられるか
def corrupt(kind, r, text, how):
    if how == "数値":                         # 観客数か売上高の数値に 7 を足す
        m = re.search(r"\d+(?=人|億円)", norm(text))
        return norm(text)[:m.start()] + str(int(m.group()) + 7) + norm(text)[m.end():]
    if kind == "試合":                        # 勝ったチームと負けたチームを入れ替える
        hs, vs = r["ホームの得点"], r["ビジターの得点"]
        win, lose = (r["ホーム"], r["ビジター"]) if hs > vs else (r["ビジター"], r["ホーム"])
        return text.replace(f"{win}が", "@").replace(f"で{lose}に", f"で{win}に").replace("@", f"{lose}が")
    return text.translate(str.maketrans({"増": "減", "減": "増"}))   # 増収と減収、増益と減益を入れ替える

for how in ["数値", "勝敗・増減"]:
    found = [bool(check(k, r, corrupt(k, r, t, how))) for (k, r), t in zip(records, template_texts)]
    print(f"わざと入れた誤り（{how}）: {sum(found)}/{len(found)} 件で検出")

テンプレートで誤りとされたもの: []
わざと入れた誤り（数値）: 24/24 件で検出
わざと入れた誤り（勝敗・増減）: 24/24 件で検出


In [6]:
import pandas as pd

# 3 つの方法の文章を検査する
results = {"テンプレート": template_texts, **llm_texts}
rows = []
for method, texts in results.items():
    errs = [check(k, r, t) for (k, r), t in zip(records, texts)]
    for kind in ["試合", "決算"]:
        idx = [i for i, (k, _) in enumerate(records) if k == kind]
        flat = [e for i in idx for e in errs[i]]
        rows.append({"方法": method, "題材": kind, "誤りを含む文章": f"{sum(bool(errs[i]) for i in idx)}/{len(idx)}",
                     "数値": sum(e.startswith("表にない数値") for e in flat), "固有名詞": sum(e.startswith("表にない固有名詞") for e in flat),
                     "勝敗・増減・スコア": sum(not e.startswith("表にない") for e in flat),
                     "平均字数": round(np.mean([len(texts[i]) for i in idx]))})
pd.DataFrame(rows)

,方法,題材,誤りを含む文章,数値,固有名詞,勝敗・増減・スコア,平均字数
0,テンプレート,試合,0/12,0,0,0,110
1,テンプレート,決算,0/12,0,0,0,92
2,LLM（表だけ）,試合,0/12,0,0,0,173
3,LLM（表だけ）,決算,10/12,14,0,2,178
4,LLM（計算済みの値も渡す）,試合,5/12,1,0,5,174
5,LLM（計算済みの値も渡す）,決算,3/12,1,0,2,182


In [7]:
# 誤りとされた箇所を読む（検査器の誤検出もありうるので、人が確かめる）
for method, texts in llm_texts.items():
    print(f"■ {method}")
    for i, ((k, r), t) in enumerate(zip(records, texts)):
        if errs := check(k, r, t):
            print(f"[{i}] {t.replace(chr(10), ' ')[:150]}\n     → {errs}")
    print()

■ LLM（表だけ）
[13] みなと食品は2025年度上期の決算を発表し、売上高が前年同期比で3.2%減少して1400億円となったものの、営業利益は20.6%増の208億円を記録しました。主な要因として同社は値上げの効果を挙げており、特に主力商品の価格上昇が寄与したと説明しています。これにより、みなと食品は業績改善を示し、今後の
     → ['表にない数値 3.2', '表にない数値 20.6']
[14] 青葉電子は2025年度上期の決算を発表し、売上高が前年同期比で減少したものの、堅調な利益を確保した。今回の決算において、青葉電子の売上高は797億円となり、前年同期の835億円からやや減少したが、これは主に円高による海外売上の影響である。しかし、営業利益は90億円と、前年同期の98億円から改善し、効
     → ['利益の増減が逆 「営業利益は90億円と、前年同期の98億円から改善し、効率的」']
[16] 東雲化学は2025年度上期の決算を発表し、売上高が前年同期比で増加したものの、円高の影響により海外売上が減少したことが報告された。今回の決算において、東雲化学の売上高は2804億円に達し、前年同期の2680億円から2.5%の増加を見せた。しかし、営業利益は167億円となり、前年同期の279億円から大
     → ['表にない数値 2.5', '売上の増減が逆 「売上が減少したことが報告された。」']
[17] ひかり通信は2025年度上期の決算を発表し、売上高が前年同期比で1.1%増加して1910億円となった。特に注目すべきは営業利益で、前年同期から微減したものの190億円を記録し、堅調な業績を維持している。主な要因として原材料費の高騰が挙げられるが、同社は効率的なコスト管理により、利益確保に成功した。
     → ['表にない数値 1.1']
[18] 若葉製薬は2025年度上期の決算を発表し、売上高が前年度同期比で7.5%増加し2509億円となった。しかし、国内市場の需要低迷により、営業利益は同時期から19.6%減少し201億円にとどまった。主な要因として、同社は医薬品市場の競争激化と需要の鈍化を挙げている。前年同期の売上高は2246億円、営業利
     → ['表にない数値 7.5', '表にない数値 19.6']
[19] 白砂建設は2

- data-to-text（表や数値データからの文章の生成）は、天気予報・試合の速報・決算の短信のように、型の決まった記事を大量に作る場面で使われる。テンプレートは表の値をそのまま流し込むので誤りがない（上の 24 件で検査の誤りは 0）が、どの記事も同じ文の型になる。LLM は言い回しを変えた読みやすい文章を書くが、長さは 1.6〜2 倍になり、表にない数値や内容を書くことがある。
- 表だけを渡した LLM は、試合の得点・日付・観客数は正しく書いた（検査で誤りは 0。読んで確かめると、勝利投手を負けたチームの投手と書いた文章が 1 件あった）。一方、決算では前年同期比を自分で計算して書いた 14 か所が、すべて誤っていた（9.8% 減を「3.2% 減少」、17.2% 増を「25.9% 増」など）。前年同期の 98 億円から 90 億円に減った営業利益を「改善し」と書いた文章もある。小さい LLM に計算をさせてはいけない。
- 計算した値をコードで足して渡すと、前年同期比の誤りはなくなった（すべて表の値を写した）。ところが試合では、足した「点差」を勝ったチームの得点と取り違え、8 対 3 の試合を「5 対 3 で下し」と書くような誤りが 5 件出た。増減額の 140 億円を「40 億円」と書いた文章も 1 件ある。値を足すほど取り違える候補が増えるので、渡す値は文章に必要なものに絞り、項目名もまぎれないものにする。数値は全部テンプレートで書き、LLM には前後の言い回しだけを任せる組み合わせも考えられる（試していない）。
- 検査器は、文中の数値を表の値と、表から計算できる値（点差、増減額、前年同期比を丸めた値、「約」や「3万2千人」のような概数）に突き合わせ、勝ち負けと増減の向き、スコアの組を表と比べる。わざと入れた誤りは 24/24 件で見つけ、テンプレートの文章では誤検出がなかった。LLM の文章では、誤りとされた 25 か所のうち 4 か所が誤検出だった（要因の「海外売上が減少」を売上高の減少とみなした 2 件、「改善が急務」を増益とみなした 1 件、「重要な 1 勝」の 1 を表にない数値とした 1 件）。
- 検査器が見逃すものもある。上の勝利投手の取り違えや、売上高が増えたのに「これらの数値を下回った」と書いた文のような関係の誤り、「首位キープ」「連勝街道を驀進」「為替差損が主な原因」のような表にない内容は、数値と固有名詞の照合では見つからない。こうした主張には、この章の「要約が原文と食い違っていないか検査する」の含意 (NLI) の判定を表を文にしたものに使うか、人が確かめる。固有名詞の照合で見つかった誤りはなかった（架空のカタカナのチーム名は、形態素解析で固有名詞にならないことが多い）。
- 「記事にない情報は入れない」と指示しても数値や名前を作ってしまうことは、この章の「記事の見出しを生成して実際の見出しと比べる」でも見た。表について質問に答える逆向きの使い方は、質問応答の章の「表の内容について質問に答える」を参照。そこでも、集計はコードで行うのが確実である。
- 英語の data-to-text の評価用データには、WebNLG（知識グラフの 3 つ組から文）、ToTTo（Wikipedia の表の一部から文）、RotoWire（バスケットボールのボックススコアから試合の記事）がある。RotoWire は、生成した記事の数値の誤りを測る研究でよく使われる。
- 試合結果と決算はこのレシピのために乱数で作った架空のもので、チーム・会社・人名は実在しない。sarashina2.2-3b-instruct は SB Intuitions のモデルで MIT ライセンス。数値は検証の GPU で実行したもの（48 件の生成でモデルの読み込みを含めて約 2 分）で、T4 では float16 で計算するので、生成される文章が変わることがある。12 件ずつの作例で、1 モデル、貪欲法での結果なので、件数の差は目安と考える。